# Python: A world of objects: advanced (optional)

This optional notebook accompanies `10_lecture.ipynb`. It covers the more advanced OOP material: **inheritance** (is-a), **object composition** (has-a), and how we used inheritance to give the rooms and players of our text adventure hospital **inventory**.

> **Note**: work through the main lecture notebook first. You will need to be comfortable with defining a class, the `__init__()` constructor, `self`, attributes and methods.

### Setup

Run the code cells below before you start. It imports the inventory version of the hospital game and defines the function we use to play it.

In [1]:
from IPython.display import clear_output

# the inventory version of the hospital text adventure
from text_adventure.advanced_game import hospital_with_inventory

In [2]:
def play_text_adventure(adventure):
    '''
    Play your text adventure!
    '''
    print('********************************************')
    print(adventure.opening, end='\n\n')
    print(adventure.current_room.describe())
    
    while adventure.active:
        user_input = input("\nWhat do you want to do? >>> ")
        response = adventure.take_action(user_input)    

        # wipe the cell output, then redraw the whole game state
        clear_output(wait=True)
        print('********************************************')
        print(f'\n{response}')
    
    print('Game over!')

## More complex OO frameworks

### Classes are customised by Inheritance

> **A note of caution**: Over time I've learnt to be somewhat wary of complex multiple inheritance structures **in any programming language**.  Inheritance brings huge benefits in terms of code reuse, but you also need to learn good OO design principals in order to avoid unexpected dependencies in your code and avoid major rework due to small changes in a projects requirements.

Let's work with a simple patient based example first.  In the code below we create a class called `Patient`.  The class `StrokePatient` **inherits** from `Patient`.  In python's OO terminology we called `Patient` a super class and `StrokePatient` a subclass.  `StrokePatient` inherits all of the super class methods and attributes.  In effect `StrokePatient` **is-a** specialisation of `Patient`.  The code below illustrates how this works. 

In [ ]:
import random

# `Patient` is refered to as a 'super class'
class Patient:
    def __init__(self, name, occupation, age):
        self.name = name
        self.occupation = occupation
        self.age = age 
        self.triage_band = None
        
    def set_random_triage_band(self):
        '''set a random triage band 1 - 5'''
        self.triage_band = random.randint(1, 5)

In [ ]:
# subclass `StrokePatient`
class StrokePatient(Patient):
    def __init__(self, name, occupation, age, stroke_type=1):
        #call the constructor of the superclass
        super().__init__(name, occupation, age)
        self.stroke_type = stroke_type

In [ ]:
# create an instance of a `StrokePatient` and use inherited methods
random.seed(42)

new_patient = StrokePatient('Joe Blogs', 'Teacher', 45)
new_patient.set_random_triage_band()
print(new_patient.name)
print(new_patient.triage_band)

## **Has-a**: An alternative OO design.  

In the previous example `StrokePatient` **is-a** specialisation of `Patient`.  An alternative way to frame this design problem as one of **object composition** where a `StrokeCase` **has-a** `Patient`.  

This approach provides slightly more flexibility than direct inheritance. For example you can pass in a different type of object as long as it implements the same interface.  It does, however, require a bit more code to setup. 

> The code below introduces a bit of OOP syntax called a **getter property** for accessing `StrokeCase.triage_band`.  Externally to the object `triage_band` appears and behaves very much like a class attribute.  However in the implementation we can see that it isn't and we can use code and logic if we want.  Here we are simply calling the `Patient.triage_band`.

In [ ]:
# this time patient is a parameter instead of a superclass
class StrokeCase:
    def __init__(self, patient, stroke_type=1):
        self.patient = patient
        self.stroke_type = stroke_type
        
    @property
    def triage_band(self):
        return self.patient.triage_band
        
    def set_random_triage_band(self):
        self.patient.set_random_triage_band()


In [ ]:
random.seed(101)
new_patient = Patient('Joe Bloggs', 'Teacher', 45)
stroke = StrokeCase(new_patient)
stroke.set_random_triage_band()
print(stroke.triage_band)

## Using inheritance to allow a `Room` and a game player to hold inventory

Now that we understand inheritance and object composition let's take look at a modified text adventure hospital.  In this version of the code we introduce the concept of **inventory**.  Have a go at playing the game.  You will find items of inventory scattered around the rooms in the game.  You can interact with these items by issuing the `get`, `drop`, and `ex` (examine) commands.  To view what inventory you are holding issue the `inv` command.

In [ ]:
from text_adventure.advanced_game import hospital_with_inventory

In [ ]:
game = hospital_with_inventory()

In [ ]:
game

In [ ]:
#play_text_adventure(game)

### OOP Implementation

The game is now a little bit more fun than before (although that isn't saying much).  Let's take a look at how the code is implemented.  

* An item of inventory is represented by a class called `InventoryItem`.
* A `Room` and a `TextWorld` is-a `InventoryHolder`
* An object that is-a `InventoryHolder` holds references to `InventoryItem`

In other words we have used inheritance to add some abilities to `Room` and `TextWorld` to hold multiple instances of `InventoryItem`.  First we define two new classes:  `InventoryItem` and `InventoryItemHolder`.

In [ ]:
class InventoryItem:
    '''
    An item found in a text adventure world that can be picked up
    or dropped.
    '''
    def __init__(self, short_description):
        self.name = short_description
        self.long_description = ''
        self.aliases = []


    def add_alias(self, new_alias):
        '''
        Add an alias (alternative name) to the InventoryItem.
        For example if an inventory item has the short description
        'credit card' then a useful alias is 'card'.

        Parameters:
        -----------
        new_alias: str
            The alias to add.  For example, 'card'

        '''
        self.aliases.append(new_alias)

In [ ]:
class InventoryHolder:
    '''
    Encapsulates the logic for adding and removing an InventoryItem
    This simulates "picking up" and "dropping" items in a TextWorld 
    '''
    def __init__(self):
        #inventory just held in a list interally
        self.inventory = []

    def list_inventory(self):
        '''
        Return a string representation of InventoryItems held.
        '''
        msg = ''
        for item in self.inventory:
            msg += f'{item.name}\n'

        return msg

    def add_inventory(self, item):
        '''
        Add an InventoryItem
        '''
        self.inventory.append(item)

    def get_inventory(self, item_name):
        '''
        Returns an InventoryItem from Room.
        Removes the item from the Room's inventory

        Params:
        ------
        item_name: str
            Key identifying item.

        Returns
        -------
        InventoryItem

        '''
        selected_item, selected_index = self.find_inventory(item_name)
         
        #remove at index and return
        del self.inventory[selected_index]
        return selected_item

    def find_inventory(self, item_name):
        '''
        Find an inventory item and return it and its index 
        in the collection.
        
        Returns:
        -------
        Tuple: InventoryItem, int
        
        Raises:
        ------
        KeyError
            Raised when an InventoryItem without a matching alias.
        '''
        selected_item = None
        selected_index = -1
        for index, item in zip(range(len(self.inventory)), self.inventory):
            if item_name in item.aliases:
                selected_item = item
                selected_index = index
                break
    
        if selected_item == None:
            raise KeyError('You cannot do that.')  

        return selected_item, selected_index

In summary, `InventoryItem` is just a textual description of an item plus a collection of aliases.  This allows a game player to interact with the item through a number of names.  For example, a "banana" might be aliased as 'fruit' or 'banana'.

`InventoryHolder` provides all the functionality to manage a collection of `InventoryItem` objects.  So we can add, remove, and list all, items from a `InventoryHolder` object's internal collection.   

Any class that inherits from `InventoryHolder` can now manage a collection of items that they 'own'.  Neat!  In our enhanced game both `Room` and `TextWorld`  inherit this functionality.  The `TextWorld` object here used as a surrogate for a player.  The modified code listing for `Room` is provided below.

In [ ]:
class Room(InventoryHolder):
    '''
    Encapsulates a location/room within a TextWorld.

    A `Room` has a number of exits to other `Room` objects

    A `Room` is-a type of `InventoryHolder`
    '''
    def __init__(self, name):
        '''
        Room constructor
        '''
        self.name = name
        self.description = ""
        self.exits = {}
        #MODIFICATION
        super().__init__()
                

    def add_exit(self, room, direction):
        '''
        Add an exit to the room

        Params:
        ------
        room: Room
            a Room object to link 

        direction: str
            The str command to access the room
        '''
        self.exits[direction] = room

    
    def exit(self, direction):
        '''
        Exit the room in the specified direction

        Params:
        ------
        direction: str
            A command string representing the direction.
        '''
        if direction in self.exits:
            return self.exits[direction]
        else:
            raise ValueError()

    def describe(self):
        msg = self.description
        ### MODIFICATION
        if len(self.inventory) > 0:
            msg += '\nYou can also see:\n'
            msg += self.list_inventory()
        return msg

## End